In [1]:
!git clone https://github.com/Miki004/VISTA.git
%cd VISTA

fatal: destination path 'VISTA' already exists and is not an empty directory.
/workspace/VISTA


In [ ]:
!uv add pycocotools torchmetrics

In [2]:
import sys
print("Python kernel:", sys.executable)
# deve essere /venv/main/bin/python

import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

Python kernel: /venv/main/bin/python
torch version: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [4]:
import sys
print("Python eseguito dal kernel:", sys.executable)
print("Versione:", sys.version)
print()
print("sys.path:")
for p in sys.path:
    print(" ", p)

Python eseguito dal kernel: /venv/main/bin/python
Versione: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]

sys.path:
  /venv/main/lib/python312.zip
  /venv/main/lib/python3.12
  /venv/main/lib/python3.12/lib-dynload
  
  /venv/main/lib/python3.12/site-packages


In [ ]:
import sys
sys.path.insert(0, '/workspace/VISTA')

from vista.utils_fun import (
    set_seed, image_to_base64, resize_image, log,
    IGNORE_CATEGORIES, get_emergency_level
)
print("vista.utils_fun OK")

from vista.qwen import get_model
print("vista.qwen OK")

from vista.pipeline.qwen_yolo import QwenYoloPipeline
print("vista.pipeline.qwen_yolo OK")

✓ vista.utils_fun OK
✓ vista.qwen OK
✓ vista.pipeline.qwen_yolo OK


In [4]:
"""Esegue la pipeline QwenYolo su una cartella di video e salva i risultati."""
from __future__ import annotations

import csv
import argparse
import ultralytics
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO

from vista.pipeline.qwen_yolo import QwenYoloPipeline
from vista.qwen import get_model   # <-- il modulo Qwen del repo della challenge


In [ ]:
import sys
sys.path.insert(0, '/workspace/VISTA')

import csv
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO
from vista.pipeline.qwen_yolo import QwenYoloPipeline
from vista.qwen import get_model

# === CONFIG ===
VIDEOS_DIR = Path("/workspace/VISTA/workspace/vista_videos")
OUT_DIR = Path("/workspace/VISTA/workspace/vista_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = {
"device": "cuda:0",

"yolo": {
"model": "yolo12m.pt",
"iou_match_threshold": 0.3,
},

"qwen": {

"model_id": "Qwen/Qwen3-VL-8B-Instruct",
"every_n_frames": 10,
"max_new_tokens": 4096,
"image_target_size": 1024,
"system_prompt": 
    """You are an operator supervising a drone operation over an accident scene. Your task is to detect and label all relevant objects in the images. Focus on the following:

    1. Vehicles:
    - Identify and classify all vehicles, including cars, trucks, motorcycles, bicycles only if they are involved in the accident.
    - Distinguish between:
        * Vehicles involved in the accident
        * Emergency or helping vehicles

    2. People:
    - Detect all people present in the scene.
    - Describe their actions and status, including but not limited to: injured, hurt, standing, sitting, walking, running, helping others, calling for help, etc.
    - Include this information in the label.

    Output format:
    - Return a valid JSON array with bounding boxes for all detected elements in the form:
    `[{"bbox_2d": [xmin, ymin, xmax, ymax], "label": "detailed description"}, ...]`
    - Example valid response:
    `[{"bbox_2d": [10, 30, 20, 60], "label": "car involved in accident"}, {"bbox_2d": [40, 15, 52, 27], "label": "person injured, sitting"}]`
    - Ensure each object is labeled with a precise description reflecting its type and status.""",

    "user_prompt":"Detect and describe all accident-relevant objects in this frame.",
    },
}


# === LOAD MODELS (una volta sola) ===
print("Carico YOLO...")
yolo = YOLO(cfg["yolo"]["model"])

print("Carico Qwen (può richiedere alcuni minuti la prima volta)...")
qwen = get_model(cfg)

pipeline = QwenYoloPipeline(yolo_model=yolo, qwen_model=qwen, caption_stride=cfg["qwen"]["every_n_frames"])
print("✓ Pipeline pronta")

Carico YOLO...
Carico Qwen (può richiedere alcuni minuti la prima volta)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

✓ Pipeline pronta


In [2]:
# === PROCESSA I VIDEO ===
video_paths = sorted(
    p for p in VIDEOS_DIR.iterdir()
    if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}
)
print(f"Trovati {len(video_paths)} video")

csv_path = OUT_DIR / "submission.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["video_id", "track_id", "frame_start", "frame_end", "caption"])

    for video_path in video_paths:
        video_id = video_path.stem
        print(f"\n=== {video_id} ===")

        tracks = defaultdict(lambda: {"frame_start": None, "frame_end": None, "caption": None})

        for result in pipeline.process_video(str(video_path)):
            if result.frame_idx % 30 == 0:
                print(f"  frame {result.frame_idx}: {len(result.detections)} detections")
            for det in result.detections:
                if det.track_id is None:
                    continue
                rec = tracks[det.track_id]
                if rec["frame_start"] is None:
                    rec["frame_start"] = result.frame_idx
                rec["frame_end"] = result.frame_idx
                if det.caption:
                    rec["caption"] = det.caption

        for tid, rec in tracks.items():
            writer.writerow([video_id, tid, rec["frame_start"], rec["frame_end"], rec["caption"] or ""])

print(f"\n✓ Submission salvata in {csv_path}")

Trovati 1 video

=== DJI_20251205134959_0001_S ===
[INFO] Qwen inference at frame 0
[INFO] Prepared input messages for the model
[INFO] Inputs tokenized and moved to device
[INFO] Model generation completed
  frame 0: 1 detections
[INFO] Qwen inference at frame 30
[INFO] Prepared input messages for the model
[INFO] Inputs tokenized and moved to device
[INFO] Model generation completed
  frame 30: 0 detections
[INFO] Qwen inference at frame 60
[INFO] Prepared input messages for the model
[INFO] Inputs tokenized and moved to device
[INFO] Model generation completed
  frame 60: 1 detections
[INFO] Qwen inference at frame 90
[INFO] Prepared input messages for the model
[INFO] Inputs tokenized and moved to device
[INFO] Model generation completed
  frame 90: 1 detections
[INFO] Qwen inference at frame 120
[INFO] Prepared input messages for the model
[INFO] Inputs tokenized and moved to device
[INFO] Model generation completed
  frame 120: 1 detections
[INFO] Qwen inference at frame 150
[INF

In [28]:
print("Device del modello:", next(qwen.model.parameters()).device)
print("Pesi totali:", sum(p.numel() for p in qwen.model.parameters()) / 1e9, "B")

# vedi se ci sono pesi su CPU
cpu_params = [n for n, p in qwen.model.named_parameters() if p.device.type == 'cpu']
print(f"Parametri su CPU: {len(cpu_params)}")
if cpu_params:
    print("Esempi:", cpu_params[:5])

Device del modello: cuda:0
Pesi totali: 3.754622976 B
Parametri su CPU: 0


In [ ]:
!{sys.executable} -m vista.annotate_video --video input.mp4 --out annotated.mp4 --start-frame 6000 --end-frame

In [ ]:
!{sys.executable} -m vista.eval_detection --data data/VistaCrash/data.yaml \
        --split val --weights yolo12m.pt --conf 0.25